Classify Text Sentiment — Preprocess the text (lowercase, remove punctuation/stopwords, tokenize, stem or lemmatize), extract features using Bag-of-Words or TF-IDF, then train a text classification model (e.g. Naive Bayes or Logistic Regression) to predict the label. Report Accuracy and F1-score, and show the top 10 most important words per class.

### 1. Imports and Configuration
We start by importing the required libraries and downloading NLTK dependencies for text preprocessing.

In [8]:
import re
import nltk
import pandas as pd
import numpy as np
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Download necessary NLTK corpora
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

### 2. Load Dataset & Preprocessing
We will load the IMDB dataset and define a cleaning function to lower-case text, strip HTML tags/punctuation, and apply lemmatization with stopword filtering.

In [9]:
# Load dataset
df = pd.read_csv('/content/IMDB Dataset.csv')
print(f"Dataset shape: {df.shape}")

# Initialize Lemmatizer and Stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove HTML tags (frequent in IMDB like <br />)
    text = re.sub(r'<br\s*/?>', ' ', text)
    # Remove punctuation & special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize, remove stopwords, and lemmatize
    words = text.split()
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(cleaned_words)

print("Preprocessing dataset (this may take a moment)... ")
df['cleaned_review'] = df['review'].apply(preprocess_text)
display(df.head())

Dataset shape: (50000, 2)
Preprocessing dataset (this may take a moment)... 


,review,sentiment,cleaned_review
0,One of the other reviewers has mentioned that ...,positive,one reviewer mentioned watching oz episode you...
1,A wonderful little production. <br /><br />The...,positive,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,positive,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stunni...


### 3. Feature Extraction and Training
We'll split the data into training and test splits, and transform the cleaned text into numerical representation via TF-IDF before fitting a Logistic Regression model.

In [10]:
# Split into Train and Test
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_review'], df['sentiment'], test_size=0.2, random_state=42, stratify=df['sentiment']
)

# Vectorize text using TF-IDF
vectorizer = TfidfVectorizer(max_features=10000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train Classifier
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

### 4. Evaluation and Feature Importance
Finally, we evaluate the classifier accuracy and investigate the top 10 features corresponding to both sentiment categories.

In [11]:
# Evaluate
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, pos_label='positive')

print("=== Model Evaluation ===")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score (Positive): {f1:.4f}\n")
print(classification_report(y_test, y_pred))

# Extract Top 10 words per class
feature_names = np.array(vectorizer.get_feature_names_out())
coefficients = model.coef_[0]

top_negative_idx = np.argsort(coefficients)[:10]
top_positive_idx = np.argsort(coefficients)[-10:][::-1]

print("=== Top 10 Most Important Words Per Class ===")
print("\nNegative Class:")
for idx in top_negative_idx:
    print(f"  - {feature_names[idx]} (coeff: {coefficients[idx]:.4f})")

print("\nPositive Class:")
for idx in top_positive_idx:
    print(f"  - {feature_names[idx]} (coeff: {coefficients[idx]:.4f})")

=== Model Evaluation ===
Accuracy: 0.8919
F1-Score (Positive): 0.8930

              precision    recall  f1-score   support

    negative       0.90      0.88      0.89      5000
    positive       0.88      0.90      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000

=== Top 10 Most Important Words Per Class ===

Negative Class:
  - worst (coeff: -10.4918)
  - waste (coeff: -8.1907)
  - awful (coeff: -7.8823)
  - bad (coeff: -7.8356)
  - boring (coeff: -6.4025)
  - poor (coeff: -5.8132)
  - terrible (coeff: -5.7450)
  - nothing (coeff: -5.3332)
  - horrible (coeff: -5.1452)
  - worse (coeff: -5.1337)

Positive Class:
  - great (coeff: 7.3608)
  - excellent (coeff: 6.8789)
  - perfect (coeff: 5.4527)
  - amazing (coeff: 5.0427)
  - best (coeff: 4.9339)
  - loved (coeff: 4.6577)
  - wonderful (coeff: 4.6228)
  - favorite (coeff: 4.6115)
  - hilarious (coeff: 4